In [3]:
# 下载输入文件（莎士比亚作品全文）
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [6]:
# 打开输入文件
with open('./data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# 查看输入文件的长度
print(f'length of dataset in character is {len(text)}')

length of dataset in character is 1115394


In [7]:
# 先统计出现了多少种字符
vocabulary = sorted(list(set(text)))
print(f'the size of vacabulary is {len(vocabulary)}')
print(f'what is in vocabulary : {vocabulary}')

the size of vacabulary is 65
what is in vocabulary : ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [8]:
# Tokenizer，将输入映射成为一张词汇表，由编码器和解码器组成

## 1. 构建String To Integer 的映射字典
stoi = { ch : i for i, ch in enumerate(vocabulary) }
## 2. 构建Integer To String 的映射字段
itos = { i : ch for i, ch in enumerate(vocabulary) }

## 3. Encoder 将输入的字符串转化为Token List
def encoder(x):
    return [stoi[ch] for ch in x]

## 4. Decoder 将Token List 转化为人能读的字符串
def decoder(digits_list):
    return ''.join([itos[i] for i in digits_list])

In [9]:
encoded_text = encoder('Hello, World!')
print(f'Encoder {'Hello, World!'} to integer list is {encoded_text}')
print(f'Deocde {encoded_text} to text is {decoder(encoded_text)}')

Encoder Hello, World! to integer list is [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
Deocde [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2] to text is Hello, World!


In [10]:
# 将输入文件加载为张量
import torch

data = torch.tensor(encoder(text), dtype=torch.long)
print(data)

print(f'tensor shape of data : {data.shape}')
print(f'tensor type of data : {data.dtype}')

tensor([18, 47, 56,  ..., 45,  8,  0])
tensor shape of data : torch.Size([1115394])
tensor type of data : torch.int64


In [11]:
# 划分训练集和测试集
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

In [12]:
# 构造一个批次的输入
context_length = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - context_length, (batch_size,))
    x = torch.stack([data[i:i + context_length] for i in ix])
    y = torch.stack([data[i + 1: i + context_length + 1] for i in ix])

    return x, y

xb, yb = get_batch('train')
print(f'x:{xb}')
print(xb.dtype, xb.shape)
print(f'y:{yb}')
print(yb.dtype, yb.shape)

x:tensor([[58, 46, 43, 56, 10,  1, 45, 56],
        [51,  1, 44, 53, 56,  1, 39,  1],
        [21,  1, 58, 46, 47, 52, 49,  6],
        [59, 56,  1, 43, 39, 56, 57,  0]])
torch.int64 torch.Size([4, 8])
y:tensor([[46, 43, 56, 10,  1, 45, 56, 39],
        [ 1, 44, 53, 56,  1, 39,  1, 51],
        [ 1, 58, 46, 47, 52, 49,  6,  0],
        [56,  1, 43, 39, 56, 57,  0, 31]])
torch.int64 torch.Size([4, 8])


In [13]:
# 构建BigramLanguageModel
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_tale = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_tale(idx)

        if targets == None:
            return logits
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            loss = F.cross_entropy(logits, targets)

            return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits = self(idx)

            logits = logits[:, -1, :]
            prob = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(prob, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

BGLM = BigramLanguageModel(len(vocabulary))

logits, loss = BGLM(xb, yb)
print(logits.shape, loss)

print(decoder(BGLM.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65]) tensor(4.5365, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
# 设置优化器为AdamW
optimizer = torch.optim.AdamW(BGLM.parameters(), lr=1e-3)

In [15]:
batch_size = 32
for step in range(1000):
    # 获取样本
    xb, yb = get_batch('train')
    # 计算logits和loss
    logits, loss = BGLM(xb, yb)
    # 清空上一步的梯度
    optimizer.zero_grad()
    # 反向传播
    loss.backward()
    # 更新参数
    optimizer.step()
print(loss.item())

3.704136610031128


In [16]:
print(decoder(BGLM.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=400)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV-ivvKcOvi-nQGX&q-YQbm dEM?px;Akr-IESq--wIWId
RFgXTpDUgM:CK$I!uo'IBT -
j?wfy fFr.&fiqtRS.ZttxGh' a!og


In [35]:
torch.manual_seed(1337)
B, C, T = 8, 4, 32
head_size = 16

x = torch.randn((B, T, C))

key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

q = query(x) # (B, T, H)
k = key(x)
v = value(x)

wei = q @ k.transpose(-2, -1) * head_size**-0.5 # (B, T, T)
wei = wei.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1), float('-inf'))
wei = torch.softmax(wei, dim=-1)
out = wei @ v
print(out)


tensor([[[ 6.7506e-02, -3.0850e-01,  3.2231e-01,  ...,  1.4239e-01,
          -2.4112e-01, -8.1826e-02],
         [ 1.1854e-01, -2.6360e-01,  2.3990e-02,  ..., -7.4462e-02,
          -2.8154e-01, -3.5547e-01],
         [ 1.0527e-01, -3.9667e-01,  2.6134e-02,  ...,  1.3244e-01,
          -2.1348e-01, -5.6924e-02],
         ...,
         [ 9.9522e-02,  1.0813e-01,  1.8889e-03,  ..., -1.6307e-01,
          -2.3972e-02, -1.0458e-01],
         [ 3.3433e-02,  6.0582e-02,  1.7713e-02,  ..., -8.9496e-02,
          -2.4624e-02, -9.2296e-02],
         [ 3.0381e-01, -1.0690e-01,  5.3355e-02,  ..., -2.0437e-01,
          -2.1294e-01, -2.0573e-01]],

        [[ 1.7095e+00, -1.2330e-01, -7.1099e-02,  ..., -6.9615e-01,
          -2.0385e-01,  6.7074e-01],
         [ 1.3888e+00, -6.6914e-01, -8.1606e-02,  ..., -3.2705e-01,
          -4.5182e-01,  3.8396e-01],
         [ 8.1601e-01, -6.7539e-01, -1.9359e-01,  ..., -1.6268e-01,
          -4.4319e-01, -1.5846e-02],
         ...,
         [ 1.7749e-01, -5